# RoboManipBaselines Colab Rollout Demo

## 学習済みACTポリシーを動かし、動画として確認する

このノートブックでは、[RoboManipBaselines](https://github.com/isri-aist/RoboManipBaselines) の学習済みACTポリシーを使い、MuJoCoシミュレーション上でロボット操作を実行します。実行結果はRMBデータとして保存し、その中に含まれるカメラ画像をMP4動画として確認します。

01のチュートリアルでは「データセットを使って模倣学習モデルを学習する」流れを扱いました。この02では、その次の段階として、学習済みモデルを使ってロボットを動かす **rollout** を体験します。

### このノートブックで行うこと

| Step | 内容 | 目的 |
| --- | --- | --- |
| 0 | Colabランタイムを確認する | GPUが使える状態か確認する |
| 1 | RoboManipBaselinesをセットアップする | rolloutに必要な環境を準備する |
| 2 | Colab実行用の補助設定を適用する | Colab上でrolloutを試せる状態にする |
| 3 | 学習済みcheckpointを準備する | rolloutで使うACTポリシーを読み込めるようにする |
| 4 | ACTポリシーをrolloutする | 学習済みモデルでロボットを動かす |
| 5 | rollout結果を動画化する | 実行結果を目で確認する |

### rolloutとは

rolloutとは、学習済みポリシーを使って、環境の中で実際に行動を生成することです。ここでは、ACTがカメラ画像とロボット状態を見て、次に取るべき関節指令を出します。その指令でMuJoCo上のロボットを動かし、結果を保存します。

trainとrolloutの違いは、次のように考えると分かりやすいです。

| 段階 | 何をするか |
| --- | --- |
| 学習 | お手本データを見て、行動の出し方を覚える |
| rollout | 過去の経験によって鍛えられたモデルが、環境内で実際に行動する |


### このノートブックで出てくる専門用語

| 用語 | 簡単な意味 | このノートブックでの役割 |
| --- | --- | --- |
| policy（ポリシー） | 観測を入力として、次に取る行動を出すモデル | ACTがロボットの関節指令を出します |
| observation（観測） | ロボットや環境から得られる情報 | カメラ画像やロボット状態が観測に含まれます |
| action（行動） | ポリシーが出力するロボットへの指令 | 関節をどのように動かすかを表します |
| environment（環境） | ロボットが動く世界 | このシートではMuJoCo上の `MujocoUR5eCable` 環境を指します |
| MuJoCo | ロボットや物体の動きを計算する物理シミュレータ | ケーブル操作をシミュレーションします |
| checkpoint | 学習済みモデルを保存したファイル。AIが計算処理を行うためのパラメータや重みが含まれます | rollout時に学習されたACTの重みを読み込みます |
| RMBデータ | RoboManipBaselinesで使われるデータ保存形式 | rollout中の画像、状態、行動などを保存します |

最初はすべてを厳密に理解する必要はありません。まずは「policyがobservationを見てactionを出し、その結果をenvironmentで試すのがrollout」と考えると、このノートブック全体の流れを追いやすくなります。

### このノートブックで特に見てほしいこと

- checkpointを使うと、学習済みモデルを後から読み込めること
- ACTが画像とロボット状態を見て行動を出すこと
- rollout結果がRMBデータとして保存されること
- 保存されたデータから動画を取り出して、動作を確認できること

> **注意**  
> このノートブックはワークショップ用の非公式教材です。RoboManipBaselinesの開発者および公式リポジトリに、このノートブック固有の内容について問い合わせないでください。


## 0. Colabランタイムの確認

このデモはGPUランタイムでの実行を想定しています。

1. 上部メニューから **ランタイム** → **ランタイムのタイプを変更** を開く
2. **ハードウェア アクセラレータ** に **T4 GPU** または利用可能なGPUを選ぶ
3. 上から順にセルを実行する

このセルでは、Python環境とGPUの状態を確認します。`nvidia-smi` の出力にGPU名が表示されれば、GPUランタイムが有効です。

### 出力の見方

- `Python:`: Pythonのバージョンです
- `Platform:`: 実行環境のOS情報です
- `nvidia-smi`: GPUの状態を表示します

`nvidia-smi` に `Tesla T4` や `L4` などのGPU名が表示されれば、このステップは成功です。


In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

result = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
if result.returncode != 0:
    raise RuntimeError(
        "GPUと通信できませんでした。Colabのメニューから "
        "ランタイム > ランタイムのタイプを変更 > GPU を選んでください。"
    )
print(result.stdout)


## 1. RoboManipBaselinesをセットアップ

RoboManipBaselines本体と、ACT rolloutに必要な依存関係をインストールします。

このセルでは以下を行います。

- RoboManipBaselinesをGitHubから取得する
- 動画処理に必要な `ffmpeg` を入れる
- PyTorchと関連ライブラリをColab上で使える版に揃える
- ACT用の追加パッケージを入れる
- ノートブック上で動画を表示するために `mediapy` を入れる

### コマンドの読み方

Colabは、ノートブックを開くたびに新しい仮想マシンを使うことが多いです。そのため、このノートブックでは必要なライブラリを最初にまとめてインストールします。

`git clone` は、GitHub上のコードをColab環境へコピーするコマンドです。`pip install` は、Pythonからそのコードやライブラリを使えるようにするコマンドです。

このセルでは、RoboManipBaselines本体だけでなく、PyTorch、ACT、動画処理用ライブラリなども準備します。数分かかることがありますが、最後までエラーで止まらなければ問題ありません。

### 完了時の状態

Colab内の `/content/RoboManipBaselines` にRoboManipBaselinesが配置され、Pythonから `robo_manip_baselines` を使える状態になります.


In [ ]:
%%bash
set -e

cd /content

if [ ! -d RoboManipBaselines/.git ]; then
  git clone https://github.com/isri-aist/RoboManipBaselines.git --recursive
else
  cd /content/RoboManipBaselines
  git pull --ff-only
  git submodule update --init --recursive
fi

apt-get update -qq
apt-get install -y -qq ffmpeg unzip libegl1 libgl1 libosmesa6 libosmesa6-dev

python -m pip install --upgrade pip setuptools wheel
python -m pip uninstall -y torch torchvision torchaudio torchcodec || true
python -m pip install \
  torch==2.6.0+cu124 \
  torchvision==0.21.0+cu124 \
  torchaudio==2.6.0+cu124 \
  --index-url https://download.pytorch.org/whl/cu124
python -m pip install torchcodec==0.2.0 mediapy

cd /content/RoboManipBaselines
python -m pip install -e .
python -m pip install -e ".[act]"

cd /content/RoboManipBaselines/third_party/act/detr
python -m pip install -e .


## 2. Colab実行用の補助設定

ACTポリシーはカメラ画像を入力として使います。そのため、画面にウィンドウを表示しない場合でも、シミュレーション内部ではカメラ画像を生成する必要があります。

通常のLinux環境とGoogle Colabでは描画まわりの条件が少し異なるため、この章ではColab上でrolloutを試しやすくするための補助設定を行います。ここはRoboManipBaselinesや模倣学習の本質ではないので、細かい実装は理解しなくて構いません。

重要なのは、**ACT rolloutではカメラ画像が必要であり、その画像をMuJoCoから取得している** という点です。

### カメラ画像とACT

ACTは、ロボットの関節角だけでなく、カメラ画像も見て行動を決めます。たとえばケーブル操作では、ケーブルの位置や形状を画像から知る必要があります。

rolloutでは次の流れが繰り返されます。

1. MuJoCoがシミュレーション内のカメラ画像を作る
2. ACTがその画像を見る
3. ACTが次の行動を出す
4. MuJoCoがロボットを動かす

この章のコードは、1番目の「画像を作る」部分をColab上で行いやすくするための準備です。

### この章の位置づけ

この章は環境設定です。ロボット学習の理論を理解するための重要部分ではありません。実行して次へ進めば十分です。


In [ ]:
import os
from pathlib import Path

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

nvidia_icd_config_path = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not nvidia_icd_config_path.exists():
    nvidia_icd_config_path.parent.mkdir(parents=True, exist_ok=True)
    nvidia_icd_config_path.write_text(
        """{
  "file_format_version" : "1.0.0",
  "ICD" : {
    "library_path" : "libEGL_nvidia.so.0"
  }
}
"""
    )

import mujoco
import torch
import mediapy as media
from IPython.display import clear_output

clear_output()
print("MUJOCO_GL:", os.environ.get("MUJOCO_GL"))
print("PYOPENGL_PLATFORM:", os.environ.get("PYOPENGL_PLATFORM"))
print("mujoco:", mujoco.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path

mujoco_env_base = Path("/content/RoboManipBaselines/robo_manip_baselines/envs/mujoco/MujocoEnvBase.py")
text = mujoco_env_base.read_text()

old_setup = '''    def setup_camera(self):
        self.cameras = {}
        for camera_id in range(self.model.ncam):
            camera = {}
            camera_name = mujoco.mj_id2name(
                self.model, mujoco.mjtObj.mjOBJ_CAMERA, camera_id
            )
            camera["name"] = camera_name
            camera["id"] = camera_id
            camera["viewer"] = OffScreenViewer(
                self.model, self.data, width=640, height=480
            )
            # Because "/" are not allowed in HDF5 keys, replace "/" with "_" in dictionary keys
            self.cameras[camera_name.replace("/", "_")] = camera

        # This is required to automatically switch context to free camera in render()
        # https://github.com/Farama-Foundation/Gymnasium/blob/81b87efb9f011e975f3b646bab6b7871c522e15e/gymnasium/envs/mujoco/mujoco_rendering.py#L695-L697
        self.mujoco_renderer._viewers["dummy"] = None

        self._first_render = True
'''

new_setup = '''    def setup_camera(self):
        self.cameras = {}
        for camera_id in range(self.model.ncam):
            camera = {}
            camera_name = mujoco.mj_id2name(
                self.model, mujoco.mjtObj.mjOBJ_CAMERA, camera_id
            )
            camera["name"] = camera_name
            camera["id"] = camera_id
            # Because "/" are not allowed in HDF5 keys, replace "/" with "_" in dictionary keys
            self.cameras[camera_name.replace("/", "_")] = camera

        # Use MuJoCo native Renderer for Colab-compatible offscreen camera images.
        self._rmb_colab_renderer = mujoco.Renderer(self.model, height=480, width=640)
        self._first_render = True
'''

old_render = '''        # Set camera images
        info["rgb_images"] = {}
        info["depth_images"] = {}
        for camera_name, camera in self.cameras.items():
            camera["viewer"].make_context_current()
            rgb_image = camera["viewer"].render(
                render_mode="rgb_array", camera_id=camera["id"]
            )
            info["rgb_images"][camera_name] = rgb_image
            depth_image = camera["viewer"].render(
                render_mode="depth_array", camera_id=camera["id"]
            )
            # See https://github.com/google-deepmind/mujoco/blob/631b16e7ad192df936195658fe79f2ada85f755c/python/mujoco/renderer.py#L170-L178
            extent = self.model.stat.extent
            near = self.model.vis.map.znear * extent
            far = self.model.vis.map.zfar * extent
            depth_image = near / (1 - depth_image * (1 - near / far))
            info["depth_images"][camera_name] = depth_image
'''

new_render = '''        # Set camera images
        info["rgb_images"] = {}
        info["depth_images"] = {}
        renderer = self._rmb_colab_renderer
        for camera_name, camera in self.cameras.items():
            renderer.disable_depth_rendering()
            renderer.update_scene(self.data, camera=camera["id"])
            rgb_image = renderer.render()
            info["rgb_images"][camera_name] = rgb_image

            renderer.enable_depth_rendering()
            renderer.update_scene(self.data, camera=camera["id"])
            depth_image = renderer.render()
            renderer.disable_depth_rendering()
            info["depth_images"][camera_name] = depth_image
'''

changed = False
if old_setup in text:
    text = text.replace(old_setup, new_setup)
    changed = True
else:
    print("setup_camera is already configured or has a different structure.")

if old_render in text:
    text = text.replace(old_render, new_render)
    changed = True
else:
    print("camera rendering path is already configured or has a different structure.")

mujoco_env_base.write_text(text)
print("Configured camera renderer:", mujoco_env_base)
print("updated:", changed)


## 3. 学習済みACT checkpointを準備

rolloutでは、学習済みポリシーの重みを保存した **checkpoint** を使います。

このセルでは、RoboManipBaselinesで公開されている `MujocoUR5eCable` 用のACT checkpointをダウンロードし、以降のセルから参照しやすい場所へ配置します。

自分で01のノートブックを最後まで学習した場合は、その学習結果のcheckpointを使うこともできます。ワークショップでは時間短縮のため、まず公開済みcheckpointを使います。

### checkpointとは

checkpointは、学習済みモデルの保存ファイルです。モデルの「覚えた内容」、つまりニューラルネットワークの重みが入っています。

RoboManipBaselinesのrolloutでは、checkpointだけでなく、同じディレクトリにある `model_meta_info.pkl` も重要です。これは、学習時に使ったデータの種類、正規化情報、カメラ名などを記録したメタ情報です。

### 公式ドキュメントでの案内

RoboManipBaselinesの公式ドキュメントでは、ローカル環境での基本的な流れとして、以下が案内されています。

1. RoboManipBaselinesを `--recursive` 付きでcloneする
2. `pip install -e .[act]` でRoboManipBaselinesとACT用依存関係を入れる
3. `third_party/act/detr` を追加でインストールする
4. データを収集する、または公開データセットを使う
5. `Train.py Act ...` で学習する
6. `Rollout.py Act MujocoUR5eCable --checkpoint ...` でrolloutする

また、公式のlearned parametersページでは、`MujocoUR5eCable` 用のACT学習済みパラメータが公開されています。

参考:

- RoboManipBaselines Quick start: https://github.com/isri-aist/RoboManipBaselines/blob/master/doc/quick_start.md
- Learned parameters: https://github.com/isri-aist/RoboManipBaselines/blob/master/doc/learned_parameters.md

### 通常のローカル環境ではどうするか

ローカルPCや研究室サーバで実行する場合は、公開済みパラメータを通常のファイルとしてダウンロード・展開し、`Rollout.py` の `--checkpoint` に `policy_last.ckpt` または `policy_best.ckpt` のパスを指定します。

例:

```bash
# RoboManipBaselinesリポジトリ内で実行
cd robo_manip_baselines

python ./bin/Rollout.py Act MujocoUR5eCable \
  --checkpoint ./checkpoint/Act/MujocoUR5eCable/policy_last.ckpt \
  --world_idx 0
```

このとき、指定したcheckpointと同じディレクトリに `model_meta_info.pkl` があることを確認してください。`model_meta_info.pkl` がないと、モデルの入出力や正規化の情報が分からず、rolloutできません。

> **備考**  
> Colabの `/content` は一時的な作業領域なので、このノートブックではダウンロード、展開、配置をセル内で行います。通常のローカル環境では、同じ作業を手動で一度行えば十分です。

### このセルで確認すること

セルの最後に、checkpointディレクトリの中身が表示されます。`policy_last.ckpt` と `model_meta_info.pkl` が表示されれば、次のrolloutでモデルを読み込む準備ができています。


In [ ]:
%%bash
set -e

ROOT=/content/RoboManipBaselines/robo_manip_baselines
CHECKPOINT_BASE="${ROOT}/checkpoint/Act"
DOWNLOAD_DIR=/content/rmb_pretrained_act_download
ZIP_FILE=/content/rmb_pretrained_act.zip
ACT_URL='https://www.dropbox.com/scl/fo/jyrz27cd2jy8mvl8ycuy7/AN80i3Z_-0ITKptxhN160Qc?rlkey=csnob2sggx4j26c4ybfg3bjps&dl=1'

# Guard against accidentally pasted Markdown links such as [https://...](https://...).
if [[ "${ACT_URL}" == \[*\]\(*\) ]]; then
  ACT_URL=$(printf '%s' "${ACT_URL}" | sed -E 's/^\[[^]]+\]\(([^)]*)\)$/\1/')
fi

mkdir -p "${CHECKPOINT_BASE}"
rm -rf "${DOWNLOAD_DIR}" "${ZIP_FILE}"
mkdir -p "${DOWNLOAD_DIR}"

echo "Download checkpoint archive..."
curl -L --fail --retry 3 -o "${ZIP_FILE}" "${ACT_URL}"
file "${ZIP_FILE}" || true

echo "Extract checkpoint archive..."
set +e
unzip -q -o "${ZIP_FILE}" -d "${DOWNLOAD_DIR}"
UNZIP_STATUS=$?
set -e

if [ "${UNZIP_STATUS}" -ne 0 ]; then
  echo "unzip returned status ${UNZIP_STATUS}. Dropbox folder zips may emit warnings; continuing if checkpoint files were extracted."
fi

CKPT=$(find "${DOWNLOAD_DIR}" -name policy_last.ckpt | head -n 1)
if [ -z "${CKPT}" ]; then
  echo "policy_last.ckpt が見つかりませんでした。ダウンロードまたは展開に失敗している可能性があります。" >&2
  echo "Downloaded file:" >&2
  ls -lh "${ZIP_FILE}" >&2 || true
  echo "Extracted files:" >&2
  find "${DOWNLOAD_DIR}" -maxdepth 5 -type f | sed -n '1,120p' >&2
  exit 1
fi

CKPT_DIR=$(dirname "${CKPT}")
if [ ! -f "${CKPT_DIR}/model_meta_info.pkl" ]; then
  echo "model_meta_info.pkl がcheckpointと同じディレクトリに見つかりませんでした。" >&2
  find "${DOWNLOAD_DIR}" -maxdepth 5 -type f | sed -n '1,120p' >&2
  exit 1
fi

TARGET_DIR="${CHECKPOINT_BASE}/Pretrained_MujocoUR5eCable_Act"
rm -rf "${TARGET_DIR}"
mkdir -p "${TARGET_DIR}"
cp -a "${CKPT_DIR}/." "${TARGET_DIR}/"

echo "Prepared checkpoint directory: ${TARGET_DIR}"
ls -lh "${TARGET_DIR}"


## 4. 学習されたACTを実行する

ここで、学習済みACTポリシーを `MujocoUR5eCable` 環境（シミュレーション環境）で実行します。

rollout中は、以下の流れが繰り返されます。

1. MuJoCo環境からカメラ画像とロボット状態を取得する
2. ACTポリシーが次の行動を予測する
3. 予測された行動でロボットを動かす
4. 結果をRMBデータとして保存する

主なオプションの意味は以下です。

| オプション | 意味 |
| --- | --- |
| `--checkpoint` | 使用する学習済みモデルを指定する |
| `--world_idx` | シミュレーション環境の配置パターンを指定する |
| `--no_render` | 画面ウィンドウを表示しない |
| `--no_plot` | 学習モデル内部のプロットを表示しない |
| `--auto_exit` | 指定時間後または成功後に自動終了する |
| `--save_rollout` | rollout中のデータを保存する |

### ログで見るポイント

実行中には、checkpointの読み込み、ポリシー構築、rollout開始、成功または失敗の結果などが表示されます。

特に以下に注目してください。

- `Using checkpoint:`: どの学習済みモデルを使っているか
- rollout中のログ: ロボットの実行が最後まで進んでいるか
- `Saved rollout RMB data:`: 保存されたRMBデータのパス

### RMBデータとは

RMBデータは、RoboManipBaselinesで使われるデータ保存形式です。rollout中に得られたカメラ画像、ロボット状態、行動などが保存されます。

成功すると、`/content/RoboManipBaselines/robo_manip_baselines/dataset/RolloutAct_...` 以下にRMBデータが保存されます。このノートブックでは、追加の結果ファイルは作らず、保存されたRMBデータのディレクトリを直接探します。

この章は、次の2つのセルに分けています。まずcheckpointが使える状態か確認し、その後でRoboManipBaselinesの中心的な実行コマンドである `python ./bin/Rollout.py ...` を実行します。


### 4.1 実行前にcheckpointを確認する

まず、rolloutで使うcheckpointと `model_meta_info.pkl` が揃っているか確認します。Colab上でcheckpointが正しく配置されていないと、rolloutが失敗します。ローカル環境で行う際は、この確認はファイルエクスプローラーやターミナルで行うことが多いです。

このセルは準備確認です。実際にロボットを動かす中心的なコマンドは、次の `4.2` のセルで実行します。


In [ ]:
%%bash
set -e

export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl

ROOT=/content/RoboManipBaselines/robo_manip_baselines
CHECKPOINT_DIR="${ROOT}/checkpoint/Act/Pretrained_MujocoUR5eCable_Act"
CKPT="${CHECKPOINT_DIR}/policy_last.ckpt"

# 3章の公開checkpointがなければ、checkpoint/Act以下の最新checkpointを使います。
if [ ! -f "${CKPT}" ]; then
  CKPT=$(find "${ROOT}/checkpoint/Act" \( -name 'policy_best.ckpt' -o -name 'policy_last.ckpt' \) | sort | tail -n 1)
fi

if [ -z "${CKPT}" ] || [ ! -f "${CKPT}" ]; then
  echo "checkpointが見つかりませんでした。3章のcheckpoint準備、または1章の学習を確認してください。" >&2
  find "${ROOT}/checkpoint/Act" -maxdepth 4 -type f 2>/dev/null | sort | tail -n 30
  exit 1
fi

CKPT_DIR=$(dirname "${CKPT}")
if [ ! -f "${CKPT_DIR}/model_meta_info.pkl" ]; then
  echo "model_meta_info.pkl がcheckpointと同じディレクトリに見つかりませんでした: ${CKPT_DIR}" >&2
  ls -lh "${CKPT_DIR}" >&2 || true
  exit 1
fi

echo "checkpoint確認OK: ${CKPT}"
echo "model_meta_info.pkl確認OK: ${CKPT_DIR}/model_meta_info.pkl"
echo "Rendering backend: MUJOCO_GL=${MUJOCO_GL}, PYOPENGL_PLATFORM=${PYOPENGL_PLATFORM}"


### 4.2 `Rollout.py` を実行する

次のセルが、この章で最も重要な実行部分です。

`python ./bin/Rollout.py Act MujocoUR5eCable ...` は、RoboManipBaselinesに対して「ACTポリシーを `MujocoUR5eCable` 環境で実行する」と指示するコマンドです。

主な読み方は次の通りです。

- `Act`: 使用する模倣学習手法
- `MujocoUR5eCable`: 実行するロボット操作タスク
- `--checkpoint`: 読み込む学習済みモデル
- `--save_rollout`: 実行結果をRMBデータとして保存

実行が終わると、最後に `Saved rollout RMB data:` として保存先が表示されます。


In [ ]:
%%bash
set -e

# ------------------------------------------------- # 
export MUJOCO_GL=egl
export PYOPENGL_PLATFORM=egl
ROOT=/content/RoboManipBaselines/robo_manip_baselines
CHECKPOINT_DIR="${ROOT}/checkpoint/Act/Pretrained_MujocoUR5eCable_Act"
CKPT="${CHECKPOINT_DIR}/policy_last.ckpt"
if [ ! -f "${CKPT}" ]; then
  CKPT=$(find "${ROOT}/checkpoint/Act" \( -name 'policy_best.ckpt' -o -name 'policy_last.ckpt' \) | sort | tail -n 1)
fi
if [ -z "${CKPT}" ] || [ ! -f "${CKPT}" ]; then
  echo "checkpointが見つかりませんでした。直前の確認セルを実行してください。" >&2
  exit 1
fi
# ---------------------------------------------------# 


cd "${ROOT}"

################################################
python ./bin/Rollout.py Act MujocoUR5eCable \
  --checkpoint "${CKPT}" \
  --world_idx 0 \
  --no_render \
  --no_plot \
  --auto_exit \
  --max_duration 8 \
  --save_rollout
#################################################

RMB_FILE=$(find "${ROOT}/dataset" -maxdepth 3 -type d -path '*RolloutAct_MujocoUR5eCable_*' -name 'MujocoUR5eCable_world*.rmb' | sort | tail -n 1)
if [ -z "${RMB_FILE}" ]; then
  echo "rolloutは終了しましたが、保存されたRMBデータが見つかりませんでした。" >&2
  find "${ROOT}/dataset" -maxdepth 3 -type d | sort | tail -n 30
  exit 1
fi

echo "Saved rollout RMB data: ${RMB_FILE}"


## 5. rollout結果からRGB動画を取り出す

rolloutで保存されたRMBデータには、カメラ画像が動画ファイルとして含まれています。

このセルでは、保存されたRMBデータからRGB動画を取り出し、Colabで再生しやすいMP4に変換します。ここでは処理を軽くするため、点群・深度・関節プロット付きの詳細動画は作らず、ロボットの動作が分かるRGB動画だけを使います。

rolloutの結果は、数値ログだけでは直感的に分かりにくいです。動画にすると、ロボットが対象物に近づけているか、動きが滑らかか、どこで失敗しているかを見やすくなります。

### このセルの流れ

1. 保存されたrollout結果ディレクトリを探す
2. その中からRGBカメラ動画を探す
3. `ffmpeg` でColab上で再生しやすいMP4に変換する

rollout結果が見つからない場合は、前のセルのrolloutが成功しているか確認してください。


In [ ]:
%%bash
set -e

ROOT=/content/RoboManipBaselines/robo_manip_baselines
OUT_MP4=/content/rmb_rollout_visualization.mp4

RMB_FILE=$(find "${ROOT}/dataset" -maxdepth 3 -type d -path '*RolloutAct_MujocoUR5eCable_*' -name 'MujocoUR5eCable_world*.rmb' | sort | tail -n 1)

if [ -z "${RMB_FILE}" ]; then
  echo "rollout結果のRMBデータが見つかりませんでした。4章のrolloutが成功したか確認してください。" >&2
  find "${ROOT}/dataset" -maxdepth 3 -type d | sort | tail -n 30
  exit 1
fi

echo "Use rollout RMB data: ${RMB_FILE}"
echo "${RMB_FILE}" > /content/rmb_latest_rollout_data.txt

RGB_VIDEO=$(find "${RMB_FILE}" -maxdepth 1 -type f -name '*_rgb_image.rmb.mp4' | sort | grep -m 1 'front_rgb_image' || true)
if [ -z "${RGB_VIDEO}" ]; then
  RGB_VIDEO=$(find "${RMB_FILE}" -maxdepth 1 -type f -name '*_rgb_image.rmb.mp4' | sort | head -n 1)
fi

if [ -z "${RGB_VIDEO}" ]; then
  echo "RGB動画が見つかりませんでした。RMBデータの中身を確認してください: ${RMB_FILE}" >&2
  find "${RMB_FILE}" -maxdepth 1 -type f | sort
  exit 1
fi

echo "Use RGB video: ${RGB_VIDEO}"

ffmpeg -hide_banner -loglevel error -y \
  -i "${RGB_VIDEO}" \
  -vf "scale=960:-2" \
  -an -vcodec libx264 -pix_fmt yuv420p -movflags +faststart \
  "${OUT_MP4}"

ls -lh "${OUT_MP4}"


## 6. Colab上でrolloutの結果を確認する

生成したMP4をノートブック内で再生します。

動画では、学習済みACTによってロボットがどのように動いたかを確認します。成功・失敗の判定だけでなく、動きが滑らかか、対象物に近づけているか、途中で不自然な動きがないかにも注目してください。

### 見るときのポイント

- ロボットの手先がケーブルへ向かっているか
- グリッパが対象物付近で動作しているか
- ケーブルの位置が変化しているか
- 途中で大きく振動したり、不自然な停止をしていないか

動画は定量評価ではありませんが、モデルの挙動を理解するための第一歩として非常に有効です。


In [ ]:
from pathlib import Path
import mediapy as media

video_path = Path("/content/rmb_rollout_visualization.mp4")
if not video_path.exists():
    raise FileNotFoundError(video_path)

video = media.read_video(str(video_path))
media.show_video(video, fps=30)


## 7. 補足: 公開データセットepisodeを動画化する

rollout結果とは別に、公開データセットに含まれる実演episodeを動画化して確認することもできます。

これは、模倣学習で使う「お手本データ」がどのような動作になっているかを見るための補助ステップです。rollout結果と見比べると、学習済みポリシーの動きがお手本に近いかどうかを考えやすくなります。

通常の流れでは、この補足ステップは必須ではありません。

### rollout動画との違い

| 動画 | 意味 |
| --- | --- |
| rollout動画 | 学習済みポリシーが自分で動いた結果 |
| データセットepisode動画 | お手本として記録された操作 |

rollout動画がうまくいかない場合でも、お手本動画を見ることで、モデルが本来まねるべき動作を確認できます。


In [ ]:
%%bash
set -e

ROOT=/content/RoboManipBaselines/robo_manip_baselines
DATASET_ROOT="${ROOT}/dataset"
DATASET_ZIP=/content/MujocoUR5eCable_Dataset30.zip
DATASET_DIR="${DATASET_ROOT}/MujocoUR5eCable"
DATASET_URL="https://www.dropbox.com/scl/fo/sykc20cnax2scom1u8sc6/AM-zLM8dAZ5h6EQ8eDXcZic?rlkey=7icbmjc6wdqnp0tngfjqlhwoh&dl=1"

mkdir -p "${DATASET_ROOT}"
rm -rf "${DATASET_DIR}" "${DATASET_ZIP}"
mkdir -p "${DATASET_DIR}"

curl -L -o "${DATASET_ZIP}" "${DATASET_URL}"
unzip -q -o "${DATASET_ZIP}" -d "${DATASET_DIR}"

RMB_FILE=$(find "${DATASET_DIR}" -maxdepth 2 -type d -name '*.rmb' | sort | head -n 1)
if [ -z "${RMB_FILE}" ]; then
  echo "データセット内に*.rmbディレクトリが見つかりませんでした。" >&2
  exit 1
fi

RGB_VIDEO=$(find "${RMB_FILE}" -maxdepth 1 -type f -name '*_rgb_image.rmb.mp4' | sort | grep -m 1 'front_rgb_image' || true)
if [ -z "${RGB_VIDEO}" ]; then
  RGB_VIDEO=$(find "${RMB_FILE}" -maxdepth 1 -type f -name '*_rgb_image.rmb.mp4' | sort | head -n 1)
fi

if [ -z "${RGB_VIDEO}" ]; then
  echo "RGB動画が見つかりませんでした。RMBデータの中身を確認してください: ${RMB_FILE}" >&2
  find "${RMB_FILE}" -maxdepth 1 -type f | sort
  exit 1
fi

echo "Use RGB video: ${RGB_VIDEO}"

ffmpeg -hide_banner -loglevel error -y \
  -i "${RGB_VIDEO}" \
  -vf "scale=960:-2" \
  -an -vcodec libx264 -pix_fmt yuv420p -movflags +faststart \
  /content/rmb_dataset_episode_visualization.mp4

ls -lh /content/rmb_dataset_episode_visualization.mp4


In [ ]:
from pathlib import Path
import mediapy as media

video_path = Path("/content/rmb_dataset_episode_visualization.mp4")
if not video_path.exists():
    raise FileNotFoundError(video_path)

video = media.read_video(str(video_path))
media.show_video(video, fps=30)


## 8. 確認ポイント

動画化できたら、以下を確認します。

- ロボットがケーブルに近づき、把持しようとしているか
- ケーブルを動かす一連の動作になっているか
- `Rollout result: success` または `failure` のどちらになったか
- rollout結果と公開データセットの実演動画に、どのような違いがあるか
- Colabでは画面表示ではなく、保存されたRGB動画を取り出すと軽量に確認できること

### エラー時の確認

| 状況 | 確認すること |
| --- | --- |
| checkpointが見つからない | 3章のダウンロードセルが成功しているか確認する |
| rollout結果が保存されない | 4章のログにエラーがないか確認する |
| 動画が見つからない | 5章で表示されるRMBデータのパスを確認する |
| 動作が不自然 | お手本動画と見比べ、どの段階でずれているか確認する |

このノートブックの主目的は、Colab上で「学習済みポリシーが環境を動かし、その結果を動画として確認する」最小ループを体験することです。
